In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
import matplotlib.pyplot as plt
from common.models import *
import robot_tools
from common.utils import get_device
from scipy.spatial import cKDTree

In [2]:
def print_msg(msg):
    msg = "## {} ##".format(msg)
    length = len(msg) 
    msg = "\n{}\n".format(msg)
    print(length*"#" + msg + length * "#")

In [3]:
device = get_device()
print_msg(f"Using device: {device}")

########################
## Using device: cuda ##
########################


In [4]:
DIM = 2
n_layers = 20
n_hidden = 128

class PeriodicEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return torch.cat([torch.cos(x), torch.sin(x)], dim=-1)

def build_model(device):
    embedding = PeriodicEmbedding()
    model_core = select_model("sll", DIM * 2, n_layers, n_hidden)
    model = nn.Sequential(embedding, model_core).to(device)
    return model

# 2D Configuration Space NDF Experiment
This notebook evaluates periodic 2D Configuration Space Neural Distance Fields (C-Space NDF). 

**Metrics evaluated:**
- **Accuracy**: Classification accuracy (Global & Boundary-focused).
- **IoU**: Intersection over Union for collision space.
- **Chamfer-L2**: Square of Chamfer distance between predicted and ground-truth boundary sets, accounting for periodic boundary conditions.

In [5]:
def predictions(model_path, X_test, device):
    model = build_model(device)
    data = torch.load(model_path, map_location=device)
    
    state_dict = data["state_dict"] if isinstance(data, dict) and "state_dict" in data else data
    
    # Mapping keys if necessary
    model_keys = set(model.state_dict().keys())
    sd_keys = set(state_dict.keys())
    
    if sd_keys == model_keys:
        model.load_state_dict(state_dict)
    else:
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        if missing or unexpected:
            new_sd = {}
            for uk in unexpected:
                val = state_dict[uk]
                nk = uk[2:] if uk.startswith("1.") else "1." + uk
                if nk in missing: new_sd[nk] = val
            if new_sd: model.load_state_dict(new_sd, strict=False)
    
    model.eval()
    with torch.no_grad():
        X_test_t = torch.Tensor(X_test).to(device)
        preds = model(X_test_t).cpu().numpy().flatten()
    return preds

In [6]:
def calculate_accuracy(preds, Y_test):
    y_pred = np.ones_like(preds)
    y_pred[preds < 0] = -1
    return (y_pred == Y_test).mean()

def calculate_iou(preds, Y_test):
    y_pred = np.ones_like(preds)
    y_pred[preds < 0] = -1
    gt_collision = (Y_test == -1)
    pred_collision = (y_pred == -1)
    intersection = np.logical_and(gt_collision, pred_collision).sum()
    union = np.logical_or(gt_collision, pred_collision).sum()
    return intersection / union if union > 0 else 1.0

def calculate_chamfer_l2(set_a, set_b):
    if len(set_a) == 0 or len(set_b) == 0: return 1.0
    period = 2 * np.pi
    set_a = np.mod(set_a, period)
    set_b = np.mod(set_b, period)
    tree_a = cKDTree(set_a, boxsize=[period, period])
    tree_b = cKDTree(set_b, boxsize=[period, period])
    dist_a_to_b, _ = tree_b.query(set_a)
    dist_b_to_a, _ = tree_a.query(set_b)
    return (np.mean(dist_a_to_b**2) + np.mean(dist_b_to_a**2)) / 2

def extract_gt_boundary_points(world_file, res=128):
    arm_problem = robot_tools.PlanningProblem(world_file)
    ticks = np.linspace(0, 2*np.pi, res, endpoint=False)
    X, Y = np.meshgrid(ticks, ticks)
    Z = np.zeros((res, res))
    for i in range(res):
        for j in range(res):
            arm_problem.robot.set_link_angles([X[i, j], Y[i, j]])
            fp = arm_problem.robot.get_current_polygon()
            Z[i, j] = -1 if any(fp.intersect(o) for o in arm_problem.workspace.obstacles) else 1
    boundary_points = []
    step = 2 * np.pi / res
    for i in range(res):
        for j in range(res):
            next_j = (j + 1) % res
            if Z[i, j] != Z[i, next_j]:
                pt = ticks[j] + 0.5 * step
                boundary_points.append([pt % (2*np.pi), ticks[i]])
    for j in range(res):
        for i in range(res):
            next_i = (i + 1) % res
            if Z[i, j] != Z[next_i, j]:
                pt = ticks[i] + 0.5 * step
                boundary_points.append([ticks[j], pt % (2*np.pi)])
    return np.array(boundary_points) if boundary_points else np.empty((0, 2))

def extract_model_boundary_points(model, device, res=128):
    model.eval()
    ticks = np.linspace(0, 2*np.pi, res, endpoint=False)
    X, Y = np.meshgrid(ticks, ticks)
    grid_points = np.stack([X.ravel(), Y.ravel()], axis=1)
    with torch.no_grad():
        grid_points_t = torch.Tensor(grid_points).to(device)
        Z = model(grid_points_t).cpu().numpy().reshape(res, res)
    boundary_points = []
    step = 2 * np.pi / res
    for i in range(res):
        for j in range(res):
            next_j = (j + 1) % res
            if Z[i, j] * Z[i, next_j] <= 0:
                t = abs(Z[i, j]) / (abs(Z[i, j]) + abs(Z[i, next_j]) + 1e-6)
                pt = ticks[j] + t * step
                boundary_points.append([pt % (2*np.pi), ticks[i]])
    for j in range(res):
        for i in range(res):
            next_i = (i + 1) % res
            if Z[i, j] * Z[next_i, j] <= 0:
                t = abs(Z[i, j]) / (abs(Z[i, j]) + abs(Z[next_i, j]) + 1e-6)
                pt = ticks[i] + t * step
                boundary_points.append([ticks[j], pt % (2*np.pi)])
    return np.array(boundary_points) if boundary_points else np.empty((0, 2))

# Global Experiment

In [7]:
def generate_random_test_set(world_file, n_points):
    arm_problem = robot_tools.PlanningProblem(world_file)
    X_test = np.random.uniform(0, 2 * np.pi, (n_points, 2))
    Y_test = np.zeros(n_points)
    for i in range(n_points):
        arm_problem.robot.set_link_angles(X_test[i])
        fp = arm_problem.robot.get_current_polygon()
        in_collision = any(fp.intersect(o) for o in arm_problem.workspace.obstacles)
        Y_test[i] = -1 if in_collision else 1
    return X_test, Y_test

In [8]:
out_dir = os.getcwd()
world = os.path.join(out_dir, 'config/block_world.yaml')
n = 10000
n_trials = 5
models_to_test = [
    os.path.join(out_dir, "output/uniform/model_final.pt"),
    os.path.join(out_dir, "output/uniform_aabb/model_final.pt"),
    os.path.join(out_dir, "output/gaussian/model_final.pt"),
    os.path.join(out_dir, "output/gaussian_aabb/model_final.pt")
]

all_results = {m: {"acc": [], "iou": []} for m in models_to_test}
for trial in range(n_trials):
    X_test, Y_test = generate_random_test_set(world, n)
    for model_path in models_to_test:
        if not os.path.exists(model_path): continue
        preds = predictions(model_path, X_test, device)
        all_results[model_path]["acc"].append(calculate_accuracy(preds, Y_test))
        all_results[model_path]["iou"].append(calculate_iou(preds, Y_test))

print_msg("GLOBAL RESULTS")
for model_path in models_to_test:
    if not os.path.exists(model_path): continue
    name = os.path.basename(os.path.dirname(model_path))
    r = all_results[model_path]
    print(f"[{name:15}] Acc: {np.mean(r['acc'])*100:6.2f}% ± {np.std(r['acc'])*100:5.2f}%, IoU: {np.mean(r['iou']):.4f} ± {np.std(r['iou']):.4f}")

####################
## GLOBAL RESULTS ##
####################
[uniform        ] Acc:  97.88% ±  0.18%, IoU: 0.9336 ± 0.0059
[uniform_aabb   ] Acc:  99.16% ±  0.14%, IoU: 0.9732 ± 0.0049
[gaussian       ] Acc:  99.14% ±  0.11%, IoU: 0.9728 ± 0.0038
[gaussian_aabb  ] Acc:  99.55% ±  0.05%, IoU: 0.9856 ± 0.0019


## Boundary Experiment
Evaluates performance near collision boundaries.

In [9]:
def generate_boundary_test_set(world, n_points, sigma=0.03):
    arm_problem = robot_tools.PlanningProblem(world)
    lo, hi = np.array([0, 0]), np.array([2*np.pi, 2*np.pi])
    span = hi - lo
    X_boundary, Y_boundary = [], []
    while len(X_boundary) < n_points:
        q1 = np.random.uniform(0, 2*np.pi, (n_points, 2))
        noise = np.random.randn(n_points, 2) * (sigma * span)
        q2 = np.clip(q1 + noise, lo, hi)
        for i in range(n_points):
            # Label q1
            arm_problem.robot.set_link_angles(q1[i])
            c1 = any(arm_problem.robot.get_current_polygon().intersect(o) for o in arm_problem.workspace.obstacles)
            l1 = -1 if c1 else 1
            # Label q2
            arm_problem.robot.set_link_angles(q2[i])
            c2 = any(arm_problem.robot.get_current_polygon().intersect(o) for o in arm_problem.workspace.obstacles)
            l2 = -1 if c2 else 1
            if l1 != l2:
                X_boundary.extend([q1[i], q2[i]]); Y_boundary.extend([l1, l2])
    return np.array(X_boundary)[:n_points], np.array(Y_boundary)[:n_points]

In [11]:
all_results_b = {m: {"b_acc": [], "chamfer": []} for m in models_to_test}
GT_bound_pts = extract_gt_boundary_points(world, res=128)

for trial in range(n_trials):
    X_bound, Y_bound = generate_boundary_test_set(world, n // 2)
    for model_path in models_to_test:
        if not os.path.exists(model_path): continue
        b_preds = predictions(model_path, X_bound, device)
        all_results_b[model_path]["b_acc"].append(calculate_accuracy(b_preds, Y_bound))
        
        # Extract model boundary points
        model = build_model(device)
        state_dict = torch.load(model_path, map_location=device)
        sd = state_dict["state_dict"] if isinstance(state_dict, dict) and "state_dict" in state_dict else state_dict
        model.load_state_dict(sd, strict=False)
        pred_pts = extract_model_boundary_points(model, device, res=128)
        all_results_b[model_path]["chamfer"].append(calculate_chamfer_l2(GT_bound_pts, pred_pts))

print_msg("BOUNDARY RESULTS")
for model_path in models_to_test:
    if not os.path.exists(model_path): continue
    name = os.path.basename(os.path.dirname(model_path))
    r = all_results_b[model_path]
    print(f"[{name:15}] B-Acc: {np.mean(r['b_acc'])*100:6.2f}% ± {np.std(r['b_acc'])*100:5.2f}%, Chamfer: {np.mean(r['chamfer']):.6f} ± {np.std(r['chamfer']):.6f}")

######################
## BOUNDARY RESULTS ##
######################
[uniform        ] B-Acc:  83.22% ±  0.51%, Chamfer: 0.007921 ± 0.000000
[uniform_aabb   ] B-Acc:  93.47% ±  0.21%, Chamfer: 0.001100 ± 0.000000
[gaussian       ] B-Acc:  92.89% ±  0.48%, Chamfer: 0.000756 ± 0.000000
[gaussian_aabb  ] B-Acc:  95.93% ±  0.18%, Chamfer: 0.000323 ± 0.000000
